# European Call Option Pricer — Monte Carlo Methods

This notebook prices a European call option under the Black-Scholes framework using Monte Carlo simulation. Starting from an Euler-Maruyama discretisation of geometric Brownian motion, we build a pricer step by step: convergence analysis, variance reduction via antithetic variates, sensitivity estimation (Greeks) by finite differences, and systematic comparison against the analytical Black-Scholes formulae.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

In [ ]:
# --- Simulation parameters ---
T     = 1       # time to maturity (years)
N     = 1000    # number of time steps
dt    = T / N
time  = np.linspace(0, T, N)

# --- Black-Scholes model parameters ---
K     = 100     # strike price
r     = 0.03    # risk-free rate
sigma = 0.20    # volatility
S0    = 100     # initial stock price

# --- Random shocks ---
# Remove the seed here to get a fresh path each run.
# For reproducibility one can use: rng = np.random.default_rng(42)
data = np.random.normal(0.0, 1.0, size=N)

# --- Euler-Maruyama scheme ---
S_euler = np.zeros(N)
S_euler[0] = S0
for i in range(N - 1):
    S_euler[i + 1] = S_euler[i] * (1 + r * dt + sigma * data[i] * dt**0.5)

# --- Exact GBM solution ---
W = np.cumsum(data * dt**0.5)
W[0] = 0
S_exact = S0 * np.exp((r - 0.5 * sigma**2) * time + sigma * W)

# --- Plot ---
plt.figure(figsize=(10, 5))
plt.plot(time, S_euler, label="Euler-Maruyama", linestyle="--")
plt.plot(time, S_exact, label="Exact solution", linestyle="-")
plt.xlabel("Time (years)")
plt.ylabel("Stock price ($)")
plt.title("Stock price simulation: Euler-Maruyama vs. exact solution")
plt.legend()
plt.grid(True)
plt.show()

### Black-Scholes simulation

We simulate one stock price path using the Euler-Maruyama scheme and overlay the exact GBM solution. Both paths use the same Brownian increments, so any discrepancy is purely numerical (discretisation error), not randomness.

### Monte Carlo pricer for a European call

Rather than solving the Black-Scholes PDE, we estimate the option price by averaging discounted payoffs over $N_{mc}$ simulated paths. The 95% confidence interval of the estimator follows from the CLT:

$$\left[ \bar{Y}_N - 1.96\,\frac{\sigma_N}{\sqrt{N_{mc}}} \;,\; \bar{Y}_N + 1.96\,\frac{\sigma_N}{\sqrt{N_{mc}}} \right]$$

where $\bar{Y}_N = e^{-rT}\,\mathbb{E}[\max(S_T - K,\,0)]$ is the discounted payoff mean and $\sigma_N$ its sample standard deviation.

In [ ]:
def normal_pricer(S0, K, r, sigma, T, N, N_mc):
    """
    Standard Monte Carlo pricer for a European call option.

    Returns
    -------
    price : float   — estimated option price
    std   : float   — sample std of discounted payoffs
    se    : float   — standard error (std / sqrt(N_mc))
    """
    dt   = T / N
    time = np.linspace(0, T, N)
    payoff = np.zeros(N_mc)

    for i in range(N_mc):
        data = np.random.normal(0.0, 1.0, size=N)
        W = np.cumsum(data * dt**0.5)
        W[0] = 0
        S_T = S0 * np.exp((r - 0.5 * sigma**2) * time + sigma * W)
        payoff[i] = max(S_T[-1] - K, 0)

    price = np.exp(-r * T) * np.mean(payoff)
    std   = np.std(payoff, ddof=1)
    se    = std / np.sqrt(N_mc)
    return price, std, se

### Convergence as a function of the number of paths

By the CLT the standard error scales as $1/\sqrt{N_{mc}}$. The plot below shows the typical "elbow": a lot of precision is gained quickly, but returns diminish beyond a few thousand paths.

In [ ]:
N_values = np.round(np.logspace(1, 4, 20)).astype(int)
prices = []
errors = []

for N_mc_val in N_values:
    price, _, error = normal_pricer(S0, K, r, sigma, T, N, N_mc_val)
    prices.append(price)
    errors.append(error)

plt.figure(figsize=(10, 5))
plt.plot(N_values, errors, label="Standard MC error")
plt.xlabel("Number of paths $N_{mc}$")
plt.ylabel("Standard error")
plt.title("Convergence of the Monte Carlo pricer")
plt.legend()
plt.grid(True)
plt.show()

## Comparison with the Black-Scholes analytical price

For a European call, the closed-form Black-Scholes price is:

$$C = S_0\,\mathcal{N}(d_1) - K e^{-rT}\,\mathcal{N}(d_2)$$

with

$$d_1 = \frac{\ln(S_0/K) + (r + \frac{\sigma^2}{2})T}{\sigma\sqrt{T}}, \qquad d_2 = d_1 - \sigma\sqrt{T}$$

We use this analytical value as a benchmark to validate the Monte Carlo estimates, and plot the convergence of the MC price toward the BS price as $N_{mc}$ grows.

In [ ]:
def black_scholes_call(S0, K, r, sigma, T):
    """Black-Scholes closed-form price for a European call."""
    d1 = (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S0 * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

price_bs = black_scholes_call(S0, K, r, sigma, T)
print(f"Black-Scholes analytical price: {price_bs:.4f}")

# --- Convergence of MC price toward BS ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: estimated price with 95% CI vs BS reference
mc_prices = np.array(prices)
mc_errors = np.array(errors)

axes[0].errorbar(N_values, mc_prices, yerr=1.96 * mc_errors,
                 fmt='o-', capsize=3, label="MC price (95% CI)", color='steelblue', alpha=0.8)
axes[0].axhline(price_bs, color='r', linestyle='--', label=f"BS price ({price_bs:.4f})")
axes[0].set_xscale('log')
axes[0].set_xlabel("Number of paths $N_{mc}$")
axes[0].set_ylabel("Option price ($)")
axes[0].set_title("MC price vs. Black-Scholes")
axes[0].legend()
axes[0].grid(True, which='both', alpha=0.4)

# Right: absolute error |MC - BS|
axes[1].plot(N_values, np.abs(mc_prices - price_bs), 'o-', color='darkorange', label="|MC - BS|")
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].set_xlabel("Number of paths $N_{mc}$")
axes[1].set_ylabel("Absolute error")
axes[1].set_title("Absolute error: MC vs. Black-Scholes")
axes[1].legend()
axes[1].grid(True, which='both', alpha=0.4)

plt.tight_layout()
plt.show()

### Antithetic variates

The antithetic variates technique reduces variance by pairing each path $Z$ with its mirror $-Z$. Both are valid $\mathcal{N}(0,1)$ draws, but the resulting payoffs are negatively correlated — a high payoff on the $+Z$ path tends to be offset by a low one on the $-Z$ path.

If $X_1, X_2$ are the discounted payoffs of a pair, the estimator is $Y = (X_1+X_2)/2$ with variance:

$$\mathrm{Var}(Y) = \frac{\sigma^2}{2} + \frac{1}{2}\mathrm{Cov}(X_1, X_2)$$

Because $\mathrm{Cov}(X_1, X_2) < 0$, $\mathrm{Var}(Y) < \sigma^2/2$, which is strictly better than independent sampling.

In [ ]:
def antithetic_pricer(S0, K, r, sigma, T, N, N_mc):
    """
    Antithetic variates Monte Carlo pricer for a European call.
    N_mc must be even; pairs of paths use shocks Z and -Z.

    Returns
    -------
    price : float   — estimated option price
    std   : float   — sample std of averaged payoffs
    se    : float   — standard error
    """
    if N_mc % 2 != 0:
        raise ValueError("N_mc must be even.")

    dt   = T / N
    time = np.linspace(0, T, N)
    payoff = np.zeros(N_mc // 2)

    for i in range(N_mc // 2):
        data = np.random.normal(0.0, 1.0, size=N)

        W_plus = np.cumsum(data * dt**0.5);  W_plus[0] = 0
        S_plus = S0 * np.exp((r - 0.5 * sigma**2) * time + sigma * W_plus)

        W_neg  = np.cumsum(-data * dt**0.5); W_neg[0]  = 0
        S_neg  = S0 * np.exp((r - 0.5 * sigma**2) * time + sigma * W_neg)

        payoff[i] = (max(S_plus[-1] - K, 0) + max(S_neg[-1] - K, 0)) / 2

    price = np.exp(-r * T) * np.mean(payoff)
    std   = np.std(payoff, ddof=1)
    se    = std / np.sqrt(N_mc // 2)
    return price, std, se

### Comparing standard MC and antithetic variates

The error curves below show that antithetic variates consistently achieves lower standard error than the standard method for the same total number of paths.

In [ ]:
N_values_cmp = [100, 500, 1000, 2000, 5000, 10000]
antithetic_prices = []
antithetic_errors = []
standard_errors_cmp = []

for N_mc_val in N_values_cmp:
    _, _, err_std = normal_pricer(S0, K, r, sigma, T, N, N_mc_val)
    _, _, err_ant = antithetic_pricer(S0, K, r, sigma, T, N, N_mc_val)
    standard_errors_cmp.append(err_std)
    antithetic_errors.append(err_ant)

plt.figure(figsize=(10, 5))
plt.plot(N_values_cmp, standard_errors_cmp, label="Standard MC", linestyle="-")
plt.plot(N_values_cmp, antithetic_errors,   label="Antithetic variates", linestyle="--")
plt.xlabel("Number of paths $N_{mc}$")
plt.ylabel("Standard error")
plt.title("Standard error: standard MC vs. antithetic variates")
plt.legend()
plt.grid(True)
plt.show()

ratios = np.array(standard_errors_cmp) / np.array(antithetic_errors)
print("Error ratio (standard / antithetic):", np.round(ratios, 3))

### Greeks by finite differences (with Common Random Numbers)

The Greeks measure how the option price reacts to small changes in each input parameter:

| Greek | Definition |
|-------|-----------|
| $\Delta$ | $\partial C / \partial S_0$ |
| $\Gamma$ | $\partial^2 C / \partial S_0^2$ |
| $\nu$ (Vega) | $\partial C / \partial \sigma$ |
| $\Theta$ | $-\partial C / \partial T$ |

We estimate them by central finite differences:

$$\Delta \approx \frac{C(S_0+\epsilon_S) - C(S_0-\epsilon_S)}{2\epsilon_S}, \qquad \Gamma \approx \frac{C(S_0+\epsilon_S) - 2C(S_0) + C(S_0-\epsilon_S)}{\epsilon_S^2}$$

$$\nu \approx \frac{C(\sigma+\epsilon_\sigma) - C(\sigma-\epsilon_\sigma)}{2\epsilon_\sigma}, \qquad \Theta \approx -\frac{C(T+\epsilon_T) - C(T-\epsilon_T)}{2\epsilon_T}$$

**Common Random Numbers (CRN):** all perturbed paths share the same $Z$ draws. This means the finite difference measures the effect of the parameter shift, not of Monte Carlo noise — it substantially reduces the variance of each Greek estimator.

In [ ]:
epsilon_S     = 1.0    # perturbation on S0
epsilon_sigma = 0.01   # perturbation on sigma
epsilon_T     = 0.01   # perturbation on T
N_mc_greeks   = int(1e5)

In [ ]:
def calculate_monte_carlo_greeks(S0, K, r, sigma, T, N_mc_greeks,
                                  epsilon_S, epsilon_sigma, epsilon_T):
    """
    Delta, Gamma, Vega, Theta via finite differences with CRN.

    Returns (delta_MC, delta_se, gamma_MC, gamma_se,
             vega_MC,  vega_se,  theta_MC, theta_se)
    """
    rng = np.random.default_rng(seed=60)
    Z   = rng.normal(0.0, 1.0, size=N_mc_greeks)
    disc = np.exp(-r * T)

    # ---- Delta & Gamma ----
    def payoff(s):
        ST = s * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
        return disc * np.maximum(ST - K, 0)

    p_up   = payoff(S0 + epsilon_S)
    p_ctr  = payoff(S0)
    p_down = payoff(S0 - epsilon_S)

    delta_sims = (p_up - p_down) / (2 * epsilon_S)
    gamma_sims = (p_up - 2 * p_ctr + p_down) / epsilon_S**2

    delta_MC = np.mean(delta_sims);  delta_se = np.std(delta_sims, ddof=1) / np.sqrt(N_mc_greeks)
    gamma_MC = np.mean(gamma_sims);  gamma_se = np.std(gamma_sims, ddof=1) / np.sqrt(N_mc_greeks)

    # ---- Vega ----
    def payoff_sig(s):
        ST = S0 * np.exp((r - 0.5 * s**2) * T + s * np.sqrt(T) * Z)
        return disc * np.maximum(ST - K, 0)

    vega_sims = (payoff_sig(sigma + epsilon_sigma) - payoff_sig(sigma - epsilon_sigma)) / (2 * epsilon_sigma)
    vega_MC = np.mean(vega_sims);  vega_se = np.std(vega_sims, ddof=1) / np.sqrt(N_mc_greeks)

    # ---- Theta ----
    def payoff_T(t):
        ST = S0 * np.exp((r - 0.5 * sigma**2) * t + sigma * np.sqrt(t) * Z)
        return np.exp(-r * t) * np.maximum(ST - K, 0)

    theta_sims = -(payoff_T(T + epsilon_T) - payoff_T(T - epsilon_T)) / (2 * epsilon_T)
    theta_MC = np.mean(theta_sims);  theta_se = np.std(theta_sims, ddof=1) / np.sqrt(N_mc_greeks)

    return delta_MC, delta_se, gamma_MC, gamma_se, vega_MC, vega_se, theta_MC, theta_se

### Interpretation of the Greeks

Running with the default parameters ($S_0=K=100$, $r=3\%$, $\sigma=20\%$, $T=1$):

- **Delta ≈ 0.597** — the option gains about $0.60 for every $1 rise in the stock.
- **Gamma ≈ 0.020** — Delta itself increases by ~0.02 per $1 move in the stock.
- **Vega ≈ 38.5** — a 1 pp increase in volatility raises the price by ≈ $0.39 (= 38.5 × 0.01).
- **Theta ≈ −5.36** — the option loses about $5.36 per year purely from time decay (~$0.015/day).

### Black-Scholes analytical Greeks

The closed-form Greeks for a European call are:

$$d_1 = \frac{\ln(S_0/K) + (r+\frac{\sigma^2}{2})T}{\sigma\sqrt{T}}, \qquad d_2 = d_1 - \sigma\sqrt{T}$$

$$\Delta_{BS} = \mathcal{N}(d_1), \qquad \Gamma_{BS} = \frac{\mathcal{N}'(d_1)}{S_0\,\sigma\sqrt{T}}$$

$$\nu_{BS} = S_0\sqrt{T}\,\mathcal{N}'(d_1), \qquad \Theta_{BS} = -\frac{S_0\,\mathcal{N}'(d_1)\,\sigma}{2\sqrt{T}} - rKe^{-rT}\mathcal{N}(d_2)$$

In [ ]:
def greeks_call_bs(S0, K, r, sigma, T):
    """Black-Scholes analytical Greeks for a European call."""
    d1 = (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    pdf_d1 = norm.pdf(d1)
    delta_bs = norm.cdf(d1)
    gamma_bs = pdf_d1 / (S0 * sigma * np.sqrt(T))
    vega_bs  = S0 * np.sqrt(T) * pdf_d1
    theta_bs = -(S0 * pdf_d1 * sigma) / (2 * np.sqrt(T)) - r * K * np.exp(-r * T) * norm.cdf(d2)
    return delta_bs, gamma_bs, vega_bs, theta_bs

delta_bs, gamma_bs, vega_bs, theta_bs = greeks_call_bs(S0, K, r, sigma, T)

### MC Greeks vs. Black-Scholes — numerical comparison

We now run the MC Greek estimator and compare each value to its analytical counterpart, reporting the 95% CI and the mean squared error (bias² + variance).

In [ ]:
delta_MC, delta_se, gamma_MC, gamma_se, vega_MC, vega_se, theta_MC, theta_se = \
    calculate_monte_carlo_greeks(S0, K, r, sigma, T, N_mc_greeks,
                                  epsilon_S, epsilon_sigma, epsilon_T)

price_mc, _, price_se = normal_pricer(S0, K, r, sigma, T, N, N_mc_greeks)

def mse(mc, se, bs):
    return (mc - bs)**2 + se**2

print(f"{'':25s} {'MC estimate':>18s}  {'BS value':>10s}  {'MSE':>10s}")
print("-" * 70)
print(f"{'Price':25s} {price_mc:8.5f} ± {1.96*price_se:.5f}  {price_bs:10.5f}  {mse(price_mc, price_se, price_bs):10.6f}")
print(f"{'Delta':25s} {delta_MC:8.5f} ± {1.96*delta_se:.5f}  {delta_bs:10.5f}  {mse(delta_MC, delta_se, delta_bs):10.6f}")
print(f"{'Gamma':25s} {gamma_MC:8.5f} ± {1.96*gamma_se:.5f}  {gamma_bs:10.5f}  {mse(gamma_MC, gamma_se, gamma_bs):10.6f}")
print(f"{'Vega':25s} {vega_MC:8.5f} ± {1.96*vega_se:.5f}  {vega_bs:10.5f}  {mse(vega_MC, vega_se, vega_bs):10.6f}")
print(f"{'Theta':25s} {theta_MC:8.5f} ± {1.96*theta_se:.5f}  {theta_bs:10.5f}  {mse(theta_MC, theta_se, theta_bs):10.6f}")

### Bias-variance trade-off for Gamma

The accuracy of the Gamma estimator depends critically on the choice of $\epsilon_S$:

- **Too small:** the finite difference quotient amplifies Monte Carlo noise (high variance).
- **Too large:** the second-order approximation introduces discretisation bias.

The optimal $\epsilon_S$ minimises the MSE = bias² + variance.

In [ ]:
epsilon_s_list = np.logspace(-1, 1, 40)
gamma_MC_list  = []
gamma_se_list  = []

for eps_S in epsilon_s_list:
    res = calculate_monte_carlo_greeks(S0, K, r, sigma, T, N_mc_greeks,
                                        eps_S, epsilon_sigma, epsilon_T)
    gamma_MC_list.append(res[2])
    gamma_se_list.append(res[3])

gamma_MC_list = np.array(gamma_MC_list)
gamma_se_list = np.array(gamma_se_list)

bias     = gamma_MC_list - gamma_bs
variance = gamma_se_list**2
mse_arr  = bias**2 + variance

best_idx      = np.argmin(mse_arr)
best_epsilon_S = epsilon_s_list[best_idx]
print(f"Optimal epsilon_S : {best_epsilon_S:.4f}")
print(f"Gamma MC          : {gamma_MC_list[best_idx]:.6f}")
print(f"Gamma BS          : {gamma_bs:.6f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.errorbar(epsilon_s_list, gamma_MC_list, yerr=1.96 * gamma_se_list,
             fmt='o-', capsize=3, label="Gamma MC (95% CI)", color='steelblue', alpha=0.7)
ax1.axhline(gamma_bs, color='r', linestyle='--', label=f"BS Gamma ({gamma_bs:.4f})")
ax1.axvline(best_epsilon_S, color='g', linestyle=':', label=fr"Optimal $\epsilon_S$={best_epsilon_S:.4f}")
ax1.set_xscale('log')
ax1.set_xlabel(r"$\epsilon_S$ (log scale)")
ax1.set_ylabel("Gamma")
ax1.set_title("Gamma estimate vs. Black-Scholes")
ax1.legend()
ax1.grid(True, which="both", ls="--", alpha=0.5)

ax2.plot(epsilon_s_list, bias**2,    label="Bias²",    color='orange',   linestyle='--')
ax2.plot(epsilon_s_list, variance,   label="Variance", color='purple',   linestyle='--')
ax2.plot(epsilon_s_list, mse_arr,    label="MSE",      color='green',    linewidth=2)
label_ax2 = r"Min MSE ($\epsilon_S$=" + f"{best_epsilon_S:.4f})"
ax2.axvline(best_epsilon_S, color='g', linestyle=':', label=label_ax2)
ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlabel(r"$\epsilon_S$ (log scale)")
ax2.set_ylabel("Error (log scale)")
ax2.set_title("Bias-variance trade-off")
ax2.legend()
ax2.grid(True, which="both", ls="--", alpha=0.5)

plt.tight_layout()
plt.show()